# Data Preparation - Cleaning

In [1]:
import pandas as pd

In [2]:
titles = pd.read_csv("/Users/christianmonsalve/Desktop/CODE/Projects/Movie_Recs/data/raw/title.basics.tsv", sep="\t")

In [3]:
titles.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [4]:
titles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12219005 entries, 0 to 12219004
Data columns (total 9 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   tconst          object
 1   titleType       object
 2   primaryTitle    object
 3   originalTitle   object
 4   isAdult         int64 
 5   startYear       object
 6   endYear         object
 7   runtimeMinutes  object
 8   genres          object
dtypes: int64(1), object(8)
memory usage: 839.0+ MB


In [5]:
IMDB_TITLES_SCHEMA = {
    "tconst": "string",
    "titleType": "string",
    "primaryTitle": "string",
    "originalTitle": "string",
    "startYear": "Int32",
    "endYear": "Int32",
    "runtimeMinutes": "Int32",
    "genres": "string"
}

In [6]:
def validate_schema(df, schema):
    df = df.copy()[list(schema.keys())]
    for col, dtype in schema.items():
        if dtype == "Int32":
            non_numeric_mask = pd.to_numeric(df[col], errors='coerce').isnull()
            df.loc[non_numeric_mask, col] = None
        df[col] = df[col].astype(dtype)
    return df

imdb_titles = validate_schema(titles, IMDB_TITLES_SCHEMA)

In [7]:
imdb_titles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12219005 entries, 0 to 12219004
Data columns (total 8 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   tconst          string
 1   titleType       string
 2   primaryTitle    string
 3   originalTitle   string
 4   startYear       Int32 
 5   endYear         Int32 
 6   runtimeMinutes  Int32 
 7   genres          string
dtypes: Int32(3), string(5)
memory usage: 640.9 MB


In [8]:
assert imdb_titles["tconst"].is_unique

In [9]:
print("Missing values: \n", imdb_titles.isnull().sum())

Missing values: 
 tconst                   0
titleType                0
primaryTitle            23
originalTitle           23
startYear          1449218
endYear           12067370
runtimeMinutes     7864109
genres                 528
dtype: int64


In [10]:
imdb_titles.isnull().any()

tconst            False
titleType         False
primaryTitle       True
originalTitle      True
startYear          True
endYear            True
runtimeMinutes     True
genres             True
dtype: bool

In [11]:
def handle_nulls(df, schema):
    df = df.copy()[list(schema.keys())]
    for col, dtype in schema.items():
        if dtype == 'string':
            imputation_value = 'Unknown'
        else:
            imputation_value = df[col].median()
        df[col] = df.copy()[col].fillna(imputation_value)
    return df
    
imdb_titles = handle_nulls(imdb_titles, IMDB_TITLES_SCHEMA)

In [12]:
imdb_titles.isnull().any()

tconst            False
titleType         False
primaryTitle      False
originalTitle     False
startYear         False
endYear           False
runtimeMinutes    False
genres            False
dtype: bool

In [13]:
movies = imdb_titles[imdb_titles['titleType'] == 'movie']

In [14]:
movies.head()

,tconst,titleType,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,Miss Jerry,1894,2014,45,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,1897,2014,100,"Documentary,News,Sport"
331,tt0000335,movie,Soldiers of the Cross,Soldiers of the Cross,1900,2014,40,"Biography,Drama"
498,tt0000502,movie,Bohemios,Bohemios,1905,2014,100,\N
570,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,1906,2014,70,"Action,Adventure,Biography"


In [15]:
genre_mask = movies['genres'] == '\\N'
movies.loc[genre_mask, 'genres'] = "Unknown"

In [16]:
movies.head()

,tconst,titleType,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,Miss Jerry,1894,2014,45,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,1897,2014,100,"Documentary,News,Sport"
331,tt0000335,movie,Soldiers of the Cross,Soldiers of the Cross,1900,2014,40,"Biography,Drama"
498,tt0000502,movie,Bohemios,Bohemios,1905,2014,100,Unknown
570,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,1906,2014,70,"Action,Adventure,Biography"


In [17]:
ratings = pd.read_csv("/Users/christianmonsalve/Desktop/CODE/Projects/Movie_Recs/data/raw/title.ratings.tsv", sep="\t")

In [18]:
ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2188
1,tt0000002,5.5,308
2,tt0000003,6.5,2288
3,tt0000004,5.1,196
4,tt0000005,6.2,3019


In [19]:
assert ratings["tconst"].is_unique

In [20]:
print("Missing values: \n", ratings.isnull().sum())

Missing values: 
 tconst           0
averageRating    0
numVotes         0
dtype: int64


In [21]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1625954 entries, 0 to 1625953
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   tconst         1625954 non-null  object 
 1   averageRating  1625954 non-null  float64
 2   numVotes       1625954 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 37.2+ MB


In [22]:
ratings['tconst'] = ratings['tconst'].astype('string')
ratings['averageRating'] = ratings['averageRating'].astype('float32')
ratings['numVotes'] = ratings['numVotes'].astype('int32')

In [23]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1625954 entries, 0 to 1625953
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   tconst         1625954 non-null  string 
 1   averageRating  1625954 non-null  float32
 2   numVotes       1625954 non-null  int32  
dtypes: float32(1), int32(1), string(1)
memory usage: 24.8 MB


## Join our two datasets

In [24]:
data = movies.merge(ratings, how='left', on='tconst')

In [42]:
data.head()

,tconst,titleType,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000009,movie,Miss Jerry,Miss Jerry,1894,2014,45,Romance,5.2,232.0
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,1897,2014,100,"Documentary,News,Sport",5.3,584.0
2,tt0000335,movie,Soldiers of the Cross,Soldiers of the Cross,1900,2014,40,"Biography,Drama",5.4,67.0
3,tt0000502,movie,Bohemios,Bohemios,1905,2014,100,Unknown,3.1,26.0
4,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,1906,2014,70,"Action,Adventure,Biography",6.0,1047.0


In [45]:
data.to_csv('/Users/christianmonsalve/Desktop/CODE/Projects/Movie_Recs/data/processed/movies_cleaned.csv', index=False)

<p>Note: Some ratings may contain nulls, but that is taken care of in exploration.ipynb to better understand how to impute these missing values</p>